# 02. Freeze the split and metric helpers

Owner: Jalil-g. Issue #2. Depends on #1.

## Goal

Freeze one balanced 8,000/2,000 split. Give later notebooks checked loaders, ID-aligned metrics, and run records.

## Setup

Normal `%run ./02_split_metrics.ipynb` defines helpers only. Set `RUN_SPLIT_SETUP=1` for the explicit setup run; the flag is consumed once. No downloads or model fits here.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, os, platform, subprocess, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 42
RUN_SETUP = os.environ.pop("RUN_SPLIT_SETUP", "0") == "1"
SPLIT_PATH = ROOT / "configs/split.json"

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def value_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def save_new_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x") as handle:
        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write("\n")

def require(condition, message):
    if not condition:
        raise ValueError(message)

## Checks

Pin one successful audit. Verify its record, component artifact, and every raw file before returning rows. Another machine must run notebook 01 and supply a matching local component artifact if the original private path is absent.

In [2]:
def checked_audit(audit_path, component_path=None):
    audit_path = Path(audit_path)
    audit = json.loads(audit_path.read_text())
    require(audit["status"] == "completed", "Audit did not complete")
    require(value_hash(audit["audit"]) == audit["audit_sha256"], "Audit summary hash mismatch")
    require(value_hash(audit["config"]) == audit["config_sha256"], "Audit config hash mismatch")
    require(file_hash(ROOT / "configs/data_manifest.json") == audit["data_manifest_sha256"], "Data manifest changed")
    manifest = json.loads((ROOT / "configs/data_manifest.json").read_text())
    require(manifest["sha256"] == audit["data_sha256"], "Data identity mismatch")
    for name, expected in audit["data_sha256"].items():
        require(file_hash(ROOT / "data/raw" / name) == expected, "Raw data changed")
    entries = [(path, sha) for path, sha in audit["artifacts"].items() if path.endswith("duplicate_components.json")]
    require(len(entries) == 1, "Expected one component artifact")
    original, expected = entries[0]
    component_path = Path(component_path) if component_path is not None else ROOT / original
    require(file_hash(component_path) == expected, "Component artifact changed")
    groups = json.loads(component_path.read_text())
    require(groups["data_sha256"] == audit["data_sha256"], "Component data mismatch")
    require(groups["config"] == audit["config"], "Component config mismatch")
    return audit, groups["components"], expected

def read_raw(name):
    return pd.read_csv(ROOT / "data/raw" / name, dtype={"id": "string"})

def validate_ids(frame):
    require("id" in frame and frame.id.notna().all() and frame.id.is_unique, "Missing or duplicate IDs")
    require(frame.id.map(lambda value: isinstance(value, str) and bool(value.strip())).all(), "IDs must be nonempty strings")

## Method

Sort IDs before the seeded stratified split. If duplicate groups exist, solve exact per-class quotas over whole components using binary integer constraints. Fail if the solver cannot establish a feasible split. Never split a component.

The current audited data has no duplicate groups. The grouped path is checked separately on synthetic data. IDs only select rows; they are not features.

In [3]:
def split_ids(frame, components, validation_per_class=1000, seed=42):
    from scipy.optimize import milp, Bounds, LinearConstraint
    from scipy.sparse import csr_matrix
    validate_ids(frame)
    frame = frame.sort_values("id").reset_index(drop=True)
    require(frame.label.isin([0, 1]).all(), "Labels must be 0/1")
    ids = frame.id.tolist()
    labels = dict(zip(ids, frame.label.astype(int)))
    parent = {key: key for key in ids}
    def find(key):
        while parent[key] != key:
            parent[key] = parent[parent[key]]
            key = parent[key]
        return key
    for component in components:
        members = [row["id"] for row in component if row["file"] == "train.csv"]
        require(all(key in parent for key in members), "Unknown training ID in component")
        for key in members[1:]:
            parent[find(key)] = find(members[0])
    groups = {}
    for key in ids:
        groups.setdefault(find(key), []).append(key)
    groups = list(groups.values())
    if all(len(group) == 1 for group in groups):
        training, validation = train_test_split(ids, test_size=2 * validation_per_class,
                                                stratify=frame.label, random_state=seed)
    else:
        quotas = np.array([[sum(labels[key] == label for key in group) for group in groups] for label in [0, 1]])
        solution = milp(c=np.random.default_rng(seed).random(len(groups)),
                        integrality=np.ones(len(groups)), bounds=Bounds(0, 1),
                        constraints=LinearConstraint(csr_matrix(quotas), validation_per_class, validation_per_class),
                        options={"time_limit": 60})
        require(solution.success, "Exact grouped quotas not solved; split was not saved")
        chosen = np.rint(solution.x).astype(int)
        require(np.array_equal(quotas @ chosen, [validation_per_class] * 2), "Grouped quota mismatch")
        validation = [key for group, select in zip(groups, chosen) if select for key in group]
        training = [key for group, select in zip(groups, chosen) if not select for key in group]
    training, validation = sorted(training), sorted(validation)
    require(not set(training) & set(validation), "Split overlap")
    require(set(training) | set(validation) == set(ids), "Split coverage mismatch")
    require(all(sum(labels[key] == label for key in validation) == validation_per_class for label in [0, 1]), "Class quota mismatch")
    for group in groups:
        require(set(group) <= set(training) or set(group) <= set(validation), "Component crosses split")
    return training, validation

def eligible_ids(unlabeled, validation, components):
    excluded = set()
    for component in components:
        touches_holdout = any(row["file"] == "test.csv" or
                              (row["file"] == "train.csv" and row["id"] in set(validation)) for row in component)
        if touches_holdout:
            excluded.update(row["id"] for row in component if row["file"] == "unlabeled.csv")
    return sorted(set(unlabeled.id) - excluded)

def freeze_split(audit_path, component_path=None, target=SPLIT_PATH):
    audit_path = Path(audit_path)
    audit, components, component_sha = checked_audit(audit_path, component_path)
    train = read_raw("train.csv")
    training, validation = split_ids(train, components)
    require(len(training) == 8000 and len(validation) == 2000, "Wrong split sizes")
    payload = {"seed": SEED, "policy": "sorted string IDs; stratified; whole duplicate groups; exact class quotas; sorted output IDs",
               "train_ids": training, "validation_ids": validation,
               "eligible_unlabeled_ids": eligible_ids(read_raw("unlabeled.csv"), validation, components),
               "data_sha256": audit["data_sha256"], "audit_path": str(audit_path.relative_to(ROOT)),
               "audit_record_sha256": file_hash(audit_path), "audit_sha256": audit["audit_sha256"],
               "component_sha256": component_sha}
    payload["split_sha256"] = value_hash(payload)
    target = Path(target)
    if target.exists():
        require(json.loads(target.read_text()) == payload, "Existing split differs; refusing replacement")
    else:
        save_new_json(target, payload)
    return payload

def load_partitions(component_path=None):
    split = json.loads(SPLIT_PATH.read_text())
    require(split["split_sha256"] == value_hash({key: value for key, value in split.items() if key != "split_sha256"}), "Split checksum mismatch")
    audit_path = ROOT / split["audit_path"]
    require(file_hash(audit_path) == split["audit_record_sha256"], "Pinned audit changed")
    audit, components, component_sha = checked_audit(audit_path, component_path)
    require(split["data_sha256"] == audit["data_sha256"] and split["component_sha256"] == component_sha, "Split inputs changed")
    train, unlabeled = read_raw("train.csv"), read_raw("unlabeled.csv")
    validate_ids(train)
    validate_ids(unlabeled)
    training, validation = split["train_ids"], split["validation_ids"]
    require(len(training) == len(set(training)) == 8000 and len(validation) == len(set(validation)) == 2000, "Invalid split sizes")
    require(not set(training) & set(validation) and set(training) | set(validation) == set(train.id), "Invalid split coverage")
    indexed = train.set_index("id", drop=False)
    for keys, quota in [(training, 4000), (validation, 1000)]:
        require(indexed.loc[keys].label.value_counts().to_dict() == {0: quota, 1: quota}, "Invalid class quotas")
    for component in components:
        keys = {row["id"] for row in component if row["file"] == "train.csv"}
        require(not (keys & set(training) and keys & set(validation)), "Duplicate leakage")
    eligible = eligible_ids(unlabeled, validation, components)
    require(eligible == split["eligible_unlabeled_ids"], "Unlabeled eligibility changed")
    return {"train": indexed.loc[training].reset_index(drop=True),
            "validation": indexed.loc[validation].reset_index(drop=True),
            "unlabeled": unlabeled.set_index("id", drop=False).loc[eligible].reset_index(drop=True),
            "split": split}

def evaluate_predictions(truth, predictions):
    validate_ids(truth)
    validate_ids(predictions)
    require(set(truth.id) == set(predictions.id) and len(truth) > 0, "Prediction coverage mismatch")
    require(truth.label.notna().all() and truth.label.isin([0, 1]).all(), "Invalid truth labels")
    aligned = truth[["id", "label"]].merge(predictions[["id", "p_ai"]], on="id", validate="one_to_one", sort=False)
    require(pd.api.types.is_numeric_dtype(aligned.p_ai), "Probabilities must be numeric")
    require(np.isfinite(aligned.p_ai).all() and aligned.p_ai.between(0, 1).all(), "Invalid probabilities")
    return {"auroc": float(roc_auc_score(aligned.label, aligned.p_ai)) if aligned.label.nunique() == 2 else None,
            "accuracy_at_0_5": float(accuracy_score(aligned.label, aligned.p_ai >= .5)), "rows": len(aligned)}

def class1_probabilities(model, texts):
    classes = list(model.classes_)
    require(len(classes) == 2 and set(classes) == {0, 1}, "Expected classes 0 and 1")
    values = np.asarray(model.predict_proba(texts))
    require(values.shape == (len(texts), 2), "Wrong probability shape")
    require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(), "Invalid model probabilities")
    require(np.allclose(values.sum(axis=1), 1), "Probability rows must sum to one")
    return values[:, classes.index(1)]

def record_run(name, command, config, seed, fit_rows, eval_rows, metrics, runtime_seconds,
               artifacts=(), status="completed", error_type=None, heldout_confirmation=None):
    require(name and all(c.isalnum() or c == "_" for c in name), "Unsafe result folder")
    require(status in {"completed", "failed"}, "Invalid status")
    split = load_partitions()["split"]
    versions = {name: importlib.metadata.version(name) for name in
                ["numpy", "pandas", "scipy", "scikit-learn", "nbconvert", "ipykernel"]}
    versions["python"] = platform.python_version()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
    dirty = bool(subprocess.check_output(["git", "diff", "HEAD", "--name-only"], cwd=ROOT, text=True).strip())
    hashes = {str(Path(path).resolve().relative_to(ROOT)): file_hash(path) for path in artifacts}
    require(fit_rows >= 0 and eval_rows >= 0 and runtime_seconds >= 0, "Invalid counts or runtime")
    if fit_rows == 10000:
        require(metrics.get("validation_auroc") is None and heldout_confirmation, "Full refit needs separate held-out evidence")
    record = {"run_id": run_id, "status": status, "error_type": error_type,
              "utc": datetime.now(timezone.utc).isoformat(), "command": command,
              "git_sha": revision, "tracked_worktree_dirty": dirty, "versions": versions,
              "lock_sha256": file_hash(ROOT / "uv.lock"), "seed": seed, "config": config,
              "config_sha256": value_hash(config), "data_sha256": split["data_sha256"],
              "split_sha256": split["split_sha256"], "split_file_sha256": file_hash(SPLIT_PATH),
              "fit_rows": fit_rows, "eval_rows": eval_rows, "metrics": metrics,
              "runtime_seconds": runtime_seconds, "artifacts": hashes, "heldout_confirmation": heldout_confirmation}
    path = ROOT / "results" / name / f"{run_id}.json"
    save_new_json(path, record)
    return path

## Run

The setup run creates the split once and checks helpers with synthetic rows. Synthetic metric checks are tests, not competition scores.

In [4]:
if RUN_SETUP:
    setup_start = time.perf_counter()
    audit_path = ROOT / "results/01_data_setup/20261110T052431604106Z.json"
    split = freeze_split(audit_path)
    parts = load_partitions()
    require(freeze_split(audit_path) == split, "Repeat split differs")
    truth = pd.DataFrame({"id": ["a", "b"], "label": [0, 1]})
    predictions = pd.DataFrame({"id": ["b", "a"], "p_ai": [.9, .1]})
    require(evaluate_predictions(truth, predictions)["auroc"] == 1, "ID alignment failed")
    require(evaluate_predictions(truth.iloc[:1], predictions.iloc[1:])["auroc"] is None, "Single-class handling failed")
    rejected = 0
    for bad in [predictions.iloc[:1], pd.concat([predictions, predictions.iloc[:1]]),
                predictions.assign(p_ai=[np.nan, .1]), predictions.assign(p_ai=[1.1, .1]),
                predictions.assign(p_ai=[".9", ".1"])]:
        try:
            evaluate_predictions(truth, bad)
        except ValueError:
            rejected += 1
    require(rejected == 5, "Malformed predictions accepted")
    class ReversedClasses:
        classes_ = np.array([1, 0])
        def predict_proba(self, texts):
            return np.tile([.8, .2], (len(texts), 1))
    require(np.array_equal(class1_probabilities(ReversedClasses(), ["synthetic"]), [.8]), "Wrong class column")
    small = pd.DataFrame({"id": list("abcdefgh"), "label": [0, 0, 0, 0, 1, 1, 1, 1]})
    groups = [[{"file": "train.csv", "id": key} for key in pair] for pair in ["ab", "ef"]]
    small_train, small_val = split_ids(small, groups, validation_per_class=2)
    require(len(small_val) == 4, "Grouped split failed")
    impossible = [[{"file": "train.csv", "id": key} for key in "abcd"]]
    try:
        split_ids(small, impossible, validation_per_class=2)
    except ValueError:
        pass
    else:
        raise ValueError("Impossible group quotas accepted")
    fake_unlabeled = pd.DataFrame({"id": ["u1", "u2", "u3"]})
    exclusion_groups = [[{"file": "unlabeled.csv", "id": "u1"}, {"file": "test.csv", "id": "t"}],
                        [{"file": "unlabeled.csv", "id": "u2"}, {"file": "train.csv", "id": "a"}]]
    require(eligible_ids(fake_unlabeled, ["a"], exclusion_groups) == ["u3"], "Unlabeled exclusion failed")
    setup_seconds = time.perf_counter() - setup_start

## Results

In [5]:
if RUN_SETUP:
    display(pd.DataFrame([{"partition": key, "rows": len(parts[key]),
                           "human": int((parts[key].label == 0).sum()),
                           "AI": int((parts[key].label == 1).sum())} for key in ["train", "validation"]]))
    print(f"Eligible unlabeled rows: {len(parts['unlabeled'])}")
    print(f"Setup and checks: {setup_seconds:.3f} seconds")
    print("Synthetic checks passed. No model fitted; competition AUROC is null.")

,partition,rows,human,AI
0,train,8000,4000,4000
1,validation,2000,1000,1000


Eligible unlabeled rows: 3000
Setup and checks: 2.126 seconds
Synthetic checks passed. No model fitted; competition AUROC is null.


## Takeaway

- One fixed split, balanced in both partitions.
- Metrics join by ID and reject malformed predictions.
- Next: teammate baseline #3, then learning curve #6.

## Save

In [6]:
if RUN_SETUP:
    result_path = record_run("02_split_metrics",
        "RUN_SPLIT_SETUP=1 uv run --locked jupyter nbconvert --execute --to notebook --inplace notebooks/02_split_metrics.ipynb",
        {"validation_per_class": 1000, "training_per_class": 4000, "synthetic_checks": "passed"},
        SEED, 0, 0, {"validation_auroc": None}, setup_seconds, artifacts=[SPLIT_PATH])
    print("Saved split identity and setup record.")

Saved split identity and setup record.
